# Reproducible ECG-Based MI Prediction Framework

This notebook demonstrates a lightweight agentic framework for interpretable ECG-based Myocardial Infarction (MI) prediction using the PTB-XL dataset.

**Disclaimer:** This is a research prototype. The outputs are not clinically validated and do not replace professional medical judgment.

In [ ]:
!pip install wfdb numpy pandas scipy scikit-learn matplotlib torch tqdm

## 1. Setup and Imports

In [ ]:
import os
import json
import ast
import random
import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Download and Preprocess PTB-XL Dataset

In [ ]:
# Code to download and load the dataset
# Note: Running this will download ~3GB of data.
data_dir = 'data/ptb-xl'
os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(os.path.join(data_dir, 'ptbxl_database.csv')):
    print("Downloading PTB-XL dataset...")
    wfdb.dl_database('ptb-xl', data_dir)
else:
    print("Dataset already downloaded.")

In [ ]:
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(path, f)) for f in tqdm(df.filename_lr)]
    else:
        data = [wfdb.rdsamp(os.path.join(path, f)) for f in tqdm(df.filename_hr)]
    data = np.array([signal for signal, meta in data])
    return data

# Load dataset metadata
df = pd.read_csv(os.path.join(data_dir, 'ptbxl_database.csv'), index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# For quick demonstration in Colab, you can limit samples
# df = df.iloc[:1000]

print("Loading raw ECG data (100 Hz)...")
X = load_raw_data(df, 100, data_dir)

# Load SCP statements
agg_df = pd.read_csv(os.path.join(data_dir, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

df['diagnostic_superclass'] = df.scp_codes.apply(aggregate_diagnostic)
df['label'] = df.diagnostic_superclass.apply(lambda x: 1 if 'MI' in x else 0)

train_fold = [1, 2, 3, 4, 5, 6, 7, 8]
val_fold = [9]
test_fold = [10]

X_train = X[df.strat_fold.isin(train_fold)]
y_train = df[df.strat_fold.isin(train_fold)]['label'].values

X_val = X[df.strat_fold.isin(val_fold)]
y_val = df[df.strat_fold.isin(val_fold)]['label'].values

X_test = X[df.strat_fold.isin(test_fold)]
y_test = df[df.strat_fold.isin(test_fold)]['label'].values
df_test = df[df.strat_fold.isin(test_fold)].copy()

print("Normalizing data...")
mean = np.mean(X_train, axis=(0, 1))
std = np.std(X_train, axis=(0, 1))

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

X_train = np.transpose(X_train, (0, 2, 1))
X_val = np.transpose(X_val, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 3. Model Definition and Training

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LightweightECGNet(nn.Module):
    def __init__(self, num_classes=1, in_channels=12):
        super(LightweightECGNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
            nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        return logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LightweightECGNet().to(device)

train_loader = DataLoader(ECGDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(ECGDataset(X_val, y_val), batch_size=64, shuffle=False)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits.squeeze(), y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
    val_loss = val_loss / len(val_loader.dataset)
    print(f"Epoch {epoch+1}: Val Loss = {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')

## 4. Evaluation and Agentic Explanations

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
test_loader = DataLoader(ECGDataset(X_test, y_test), batch_size=64, shuffle=False)

all_probs = []
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        logits = model(X_batch.to(device)).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()
        if probs.ndim == 0: probs = np.array([probs])
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)

print("AUROC:", roc_auc_score(y_test, all_probs))
print("AUPRC:", average_precision_score(y_test, all_probs))

In [ ]:
df_test['predicted_probability'] = all_probs
df_test['predicted_label'] = all_preds
df_test['true_label'] = y_test

def generate_explanation(row):
    prob = row['predicted_probability']
    pred = row['predicted_label']
    true = row['true_label']
    
    if prob < 0.3: risk = "Low risk"
    elif prob < 0.7: risk = "Intermediate risk"
    else: risk = "High risk"
    
    if pred == 1 and true == 1: c_type = "True Positive"
    elif pred == 1 and true == 0: c_type = "False Positive"
    elif pred == 0 and true == 1: c_type = "False Negative"
    else: c_type = "True Negative"
        
    exp = (f"The ECG-based model predicted a {risk.lower()} of myocardial infarction "
           f"for this recording (Probability = {prob:.3f}). In retrospective analysis, "
           f"this case was a {c_type}. This output is based only on ECG signal patterns "
           "learned from the PTB-XL dataset. Since no laboratory values, biomarkers (e.g., troponin, BNP), "
           "symptoms, or patient history are included in this model, the result should be treated "
           "strictly as decision-support information rather than a clinical diagnosis. "
           "Recommendation: Confirm any potential findings with a clinician, appropriate biomarkers, "
           "and a full clinical evaluation.")
    return exp

df_test['explanation'] = df_test.apply(generate_explanation, axis=1)
print("Example Explanation:\n")
print(df_test.iloc[0]['explanation'])